# CTDDG conditional fine-tuning — PyTorch

Stage 2: loads the unconditional pretrained checkpoint, adds the protein conditioning path,
trains on BindingDB protein–ligand pairs with the Eq. 8 toxicity-weighted loss.

Ligands are labelled with `models/ames_random_forest.pkl`. Pairs are assembled directly from
the raw fold files, so no pre-built `<SMILES>\t<embedding>` file is needed.

Requires **average-pooled** ProSE embeddings in each split directory —
`<fold>_<tr|te>_unique_embeddings_bepler.h5`, one `(6165,)` vector per protein, written by
`embed_sequences.py --pool avg`. Per-residue `--pool none` output is rejected on load.

Best weights land in `outputs/finetune/<fold>/ckpt_best.pt` with a test bundle beside them.

In [ ]:
import os
import gc
import glob
import json
import math
import time
import pickle
import random
import warnings
from abc import ABCMeta, abstractmethod

import numpy as np
import pandas as pd
import networkx as nx
from scipy import sparse

import rdkit
from rdkit import Chem
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Sampler
from torch.autograd.function import once_differentiable

print(f"torch {torch.__version__} | rdkit {rdkit.__version__}")

## Config

In [ ]:
FORCE_RESTART = False
FOLD = "d1"

GPU_ID = 0
CTX = torch.device(f"cuda:{GPU_ID}") if torch.cuda.is_available() else torch.device("cpu")
print(f"Device: {CTX}")

REPO_ROOT = os.environ.get("CTDDG_REPO_ROOT", "/home/muhamed/repo/CTDDG")
ATOM_TYPES_PATH = os.path.join(REPO_ROOT, "datasets/data/atom_types.txt")
BINDINGDB_DIR = os.path.join(REPO_ROOT, "datasets/data/bindingdb")
RF_MODEL_PATH = os.path.join(REPO_ROOT, "models/ames_random_forest.pkl")
FP_CONFIG_PATH = os.path.join(REPO_ROOT, "models/fingerprint_config.pkl")
PRETRAINED_CKPT = os.path.join(REPO_ROOT, "checkpoints/ckpt.pt")

OUT_DIR = os.path.join(REPO_ROOT, "outputs/finetune", FOLD)
os.makedirs(OUT_DIR, exist_ok=True)

CKPT_BEST = os.path.join(OUT_DIR, "ckpt_best.pt")      # evaluate this one
CKPT_LAST = os.path.join(OUT_DIR, "ckpt_last.pt")      # resume from this one
CKPT_TRAINER = os.path.join(OUT_DIR, "trainer.pt")
CKPT_CONFIG = os.path.join(OUT_DIR, "configs.json")
LOG_FILE = os.path.join(OUT_DIR, "log.out")
VAL_LOG_FILE = os.path.join(OUT_DIR, "val.out")
SPLIT_MANIFEST = os.path.join(OUT_DIR, "split_manifest.json")
TEST_BUNDLE = os.path.join(OUT_DIR, "test_set.npz")

# fold -> the `4_org_*` directory suffix on disk
FOLD_SUFFIX = {"d1": "4_org_1042_104", "d2": "4_org_1000_112", "d3": "4_org_1004_122",
               "d4": "4_org_1002_103", "d5": "4_org_1036_124"}

AMES_CACHE = os.path.join(BINDINGDB_DIR, "ligand_ames_labels.csv")

# Fraction of held-out proteins used for early stopping; the rest is the untouched test half.
VAL_PROTEIN_FRACTION = 0.5
SPLIT_SEED = 42

MAX_ATOMS = 100          # preprocessing drops larger ligands
LIMIT_PAIRS = None       # e.g. 20_000 for a smoke run

BATCH_SIZE = 8
BATCH_SIZE_VAL = 8
K_PATHS = 5
P_ALPHA = 0.8
LAMBDA_TOX = 1.0
N_C = 6165               # average-pooled ProSE/Bepler width; validated per protein on load
ALLOW_N_C_OVERRIDE = False   # True = adopt the embedding file's width instead of erroring
EMB_EXTENSIONS = ('.h5', '.hdf5', '.npy', '.txt', '.csv')

F_E = 16
F_H = [32, 64, 128, 128, 256, 256]
F_SKIP = 256
F_C = [512]
FH_POLICY = 128
ACTIVATION = 'relu'
N_RNN = 3
D_RECEPTIVE = 2
MAX_ITER = 120

LR = 1e-4
DECAY = 0.002            # lr *= (1 - DECAY) every DECAY_STEP iterations
DECAY_STEP = 100
CLIP_GRAD = 3.0
SUMMARY_STEP = 200
MAX_EPOCHS = 10
PATIENCE = 10
ES_DELTA = 1e-2

# 'weighted' watches the Eq. 8 objective; 'nll' watches plain -mean(l). Switch to 'nll' if the
# toxic branch diverges, since Eq. 8's toxic term is unbounded below.
EARLY_STOP_ON = 'weighted'
TOX_FLOOR = None         # e.g. -150.0 to floor the toxic branch

NUM_WORKERS = 8
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

for p in (ATOM_TYPES_PATH, BINDINGDB_DIR, RF_MODEL_PATH, FP_CONFIG_PATH):
    assert os.path.exists(p), f"Not found: {p}"
if not os.path.exists(PRETRAINED_CKPT):
    print(f"[WARN] No pretrained checkpoint at {PRETRAINED_CKPT} — "
          f"the conditional model would start from random weights.")
print("[OK] Paths verified.")

## Fold file discovery and embedding loading

In [ ]:
def split_dir(split):
    """split: 'train' or 'test'."""
    return os.path.join(BINDINGDB_DIR, f"{split}_dataset", f"{split}_{FOLD_SUFFIX[FOLD]}")


def find_fold_files(split):
    """Locate the four files one split needs: ligands, proteins, unique proteins, embeddings."""
    d = split_dir(split)
    tag = "tr" if split == "train" else "te"

    def one(pattern, what):
        hits = sorted(glob.glob(os.path.join(d, pattern)))
        if not hits:
            raise FileNotFoundError(
                f"{what}: no match for {pattern!r} in {d}\n"
                f"  Directory holds: {sorted(os.listdir(d))[:25]}")
        return hits[0]

    ligands = one("ligands_*_corrected", "ligand pairs")
    proteins = one("proteins_*_corrected", "protein pairs")
    unique = one(f"{FOLD}_{tag}_unique_proteins.txt", "unique protein list")

    # exact name first, then any other embedding file in the directory
    expected = f"{FOLD}_{tag}_unique_embeddings_bepler.h5"
    emb = os.path.join(d, expected)
    if not os.path.isfile(emb):
        emb_hits = [p for p in sorted(glob.glob(os.path.join(d, "*embeddings*bepler*")))
                    if os.path.splitext(p)[1].lower() in EMB_EXTENSIONS]
        if not emb_hits:
            present = sorted(os.listdir(d)) if os.path.isdir(d) else []
            raise FileNotFoundError(
                f"Protein embeddings missing for fold {FOLD} ({split} split).\n"
                f"  Expected: {os.path.join(d, expected)}\n"
                f"  Directory holds: {present[:25]}\n"
                f"  Generate average-pooled embeddings with:\n"
                f"    python prose/embed_sequences.py --pool avg \\\n"
                f"        -o {expected} {FOLD}_{tag}_unique_proteins_bepler.fa\n"
                f"  --pool avg is required: it yields one ({N_C},) vector per protein. "
                f"--pool none yields per-residue (L, {N_C}) arrays, which this code rejects.")
        emb = emb_hits[0]
        print(f"[WARN] {expected} not found; using {os.path.basename(emb)} instead.")
    return ligands, proteins, unique, emb


def _as_pooled_vector(arr, name, path, width):
    """Validate one protein's embedding and return it as 1-D float32.

    `width` is the required length, or None to accept whatever the first protein set.
    """
    arr = np.asarray(arr)

    if arr.ndim != 1:
        squeezed = arr.squeeze()
        if squeezed.ndim == 1:
            arr = squeezed                       # (1, N_C) / (N_C, 1) are fine
        elif arr.ndim == 2:
            raise ValueError(
                f"{path}: protein {name!r} has shape {arr.shape} — per-residue embeddings.\n"
                f"  This file was written with `--pool none`. Fine-tuning conditions on ONE "
                f"fixed-size vector per protein, so regenerate it with:\n"
                f"    python prose/embed_sequences.py --pool avg -o <out>.h5 <in>.fa\n"
                f"  Expected per protein: ({N_C},)   got: {arr.shape}")
        else:
            raise ValueError(f"{path}: protein {name!r} has shape {arr.shape}; "
                             f"expected a 1-D ({N_C},) vector.")

    if width is not None and arr.shape[0] != width:
        hint = ""
        if arr.shape[0] % N_C == 0 and arr.shape[0] > N_C:
            hint = (f"\n  {arr.shape[0]} = {arr.shape[0] // N_C} x {N_C}, so this looks like a "
                    f"flattened per-residue (`--pool none`) embedding. Regenerate with "
                    f"`--pool avg`.")
        raise ValueError(
            f"{path}: protein {name!r} has width {arr.shape[0]}; expected {width}.{hint}")

    if arr.shape[0] == 0:
        raise ValueError(f"{path}: protein {name!r} has an empty embedding.")

    return arr.astype(np.float32, copy=False)


def _h5_key_order(f, n_expected, path):
    """Dataset keys in protein order: p1, p2, ... pN as written by embed_sequences.py."""
    keys = list(f.keys())
    if not keys:
        raise ValueError(f"{path}: HDF5 file contains no datasets.")

    if len(keys) != n_expected:
        shown = keys[:5] if len(keys) <= 10 else keys[:5] + ['...'] + keys[-5:]
        raise ValueError(
            f"{path}: {len(keys)} embedded proteins but {n_expected} unique proteins in the "
            f"fold's protein list.\n"
            f"  Keys: {shown}\n"
            f"  The two must correspond one-to-one, in order. Re-run embed_sequences.py "
            f"--pool avg on this fold's *_unique_proteins_bepler.fa.")

    named = [f"p{i + 1}" for i in range(n_expected)]
    if all(k in f for k in named):
        return named

    missing = [k for k in named if k not in f]
    def _num(k):
        digits = ''.join(ch for ch in k if ch.isdigit())
        return int(digits) if digits else -1
    print(f"[WARN] {os.path.basename(path)}: expected keys p1..p{n_expected}; "
          f"{len(missing)} absent (e.g. {missing[:8]}). Falling back to numeric key order — "
          f"check that the embeddings still line up with the protein list.")
    return sorted(keys, key=_num)


def load_embeddings(path, n_expected):
    """Read average-pooled ProSE embeddings into [n_proteins, N_C] float32.

    Row i is protein `p{i+1}`, which is the bepler FASTA header order and therefore the line
    order of `<fold>_<tr|te>_unique_proteins.txt`.
    """
    ext = os.path.splitext(path)[1].lower()
    if ext not in EMB_EXTENSIONS:
        raise ValueError(f"{path}: unsupported embedding format {ext!r}; "
                         f"expected one of {sorted(EMB_EXTENSIONS)}.")

    if ext in ('.h5', '.hdf5'):
        import h5py
        with h5py.File(path, 'r') as f:
            order = _h5_key_order(f, n_expected, path)

            # the first protein fixes the width every other protein must match
            first = _as_pooled_vector(f[order[0]][()], order[0], path,
                                      None if ALLOW_N_C_OVERRIDE else N_C)
            width = first.shape[0]
            rows = [first] + [_as_pooled_vector(f[k][()], k, path, width)
                              for k in order[1:]]
        C = np.stack(rows, axis=0)

    elif ext == '.npy':
        arr = np.load(path)
        if arr.ndim != 2:
            raise ValueError(
                f"{path}: expected a 2-D [n_proteins, {N_C}] array, got shape {arr.shape}. "
                f"Per-residue (`--pool none`) embeddings are not usable here.")
        C = arr.astype(np.float32, copy=False)

    else:
        raw = []
        with open(path) as f:
            for ln, line in enumerate(f, 1):
                parts = line.replace(',', ' ').split()
                if not parts:
                    continue
                try:
                    float(parts[0])
                except ValueError:
                    parts = parts[1:]            # leading sequence-name column
                raw.append((f"line {ln}", np.array(parts, dtype=np.float32)))
        if not raw:
            raise ValueError(f"{path}: no embedding rows parsed.")
        first = _as_pooled_vector(raw[0][1], raw[0][0], path,
                                  None if ALLOW_N_C_OVERRIDE else N_C)
        width = first.shape[0]
        C = np.stack([first] + [_as_pooled_vector(a, nm, path, width)
                                for nm, a in raw[1:]], axis=0)

    if C.shape[0] != n_expected:
        raise ValueError(
            f"{path}: {C.shape[0]} embedding rows but {n_expected} unique proteins in the "
            f"fold's protein list. The two must correspond one-to-one, in order.")

    if C.shape[1] != N_C:
        if not ALLOW_N_C_OVERRIDE:
            raise ValueError(
                f"{path}: embedding width {C.shape[1]}, expected N_C = {N_C}.\n"
                f"  Average-pooled ProSE/Bepler embeddings are {N_C}-d — either the wrong file "
                f"is being read or it was not produced with `--pool avg`.\n"
                f"  Set ALLOW_N_C_OVERRIDE = True to train at width {C.shape[1]} instead.")
        print(f"[WARN] N_C {N_C} -> {C.shape[1]} (ALLOW_N_C_OVERRIDE is on).")

    if not np.isfinite(C).all():
        n_bad = int((~np.isfinite(C)).any(axis=1).sum())
        raise ValueError(f"{path}: {n_bad} protein embeddings contain NaN or inf.")

    return C

## Ames labelling

In [ ]:
def _load_pickle(path):
    with open(path, 'rb') as f:
        try:
            return pickle.load(f)
        except Exception:
            import joblib
            return joblib.load(path)


def morgan_matrix(smiles_list, radius, n_bits):
    """[n, n_bits] uint8 Morgan fingerprints. Rows for unparseable SMILES stay zero."""
    try:
        from rdkit.Chem import rdFingerprintGenerator
        gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
        fp_of = lambda m: gen.GetFingerprint(m)
    except (ImportError, AttributeError):
        from rdkit.Chem import AllChem
        fp_of = lambda m: AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=n_bits)

    out = np.zeros((len(smiles_list), n_bits), dtype=np.uint8)
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            out[i] = np.frombuffer(fp_of(mol).ToBitString().encode(), dtype=np.uint8) - ord('0')
        if (i + 1) % 20000 == 0:
            print(f"    fingerprinted {i + 1}/{len(smiles_list)}")
    return out


def ames_labels(smiles_list, cache_path=AMES_CACHE):
    """SMILES -> {0,1} Ames mutagenicity from the repo's Random Forest. Cached across folds."""
    cache = {}
    if cache_path and os.path.exists(cache_path):
        df = pd.read_csv(cache_path)
        cache = dict(zip(df['smiles'].astype(str), df['ames'].astype(int)))
        print(f"[INFO] Ames cache: {len(cache)} entries from {cache_path}")

    todo = sorted({s for s in smiles_list if s not in cache})
    if todo:
        fp_cfg = _load_pickle(FP_CONFIG_PATH)
        radius = int(fp_cfg.get('radius', 2))
        n_bits = int(fp_cfg.get('n_bits', fp_cfg.get('nBits', 2048)))
        print(f"[INFO] Labelling {len(todo)} new ligands | {fp_cfg.get('type', 'Morgan')} "
              f"radius {radius}, {n_bits} bits")

        rf = _load_pickle(RF_MODEL_PATH)
        X = morgan_matrix(todo, radius, n_bits)
        if getattr(rf, 'n_features_in_', n_bits) != n_bits:
            raise ValueError(f"RF expects {rf.n_features_in_} features, fingerprint config "
                             f"gives {n_bits}.")
        pred = rf.predict(X).astype(int)
        cache.update(zip(todo, pred))

        if cache_path:
            pd.DataFrame({'smiles': list(cache.keys()), 'ames': list(cache.values())}) \
              .to_csv(cache_path, index=False)
            print(f"[INFO] Cache written: {cache_path}")

    labels = [int(cache[s]) for s in smiles_list]
    n_tox = sum(labels)
    print(f"[INFO] Ames: {n_tox} toxic / {len(labels) - n_tox} non-toxic "
          f"({100.0 * n_tox / max(1, len(labels)):.1f}% toxic)")
    return labels

## Molecule spec and graph traversal

Unchanged from the pretraining code — both phases must build graphs identically or the
transferred weights are meaningless.

In [ ]:
class MoleculeSpec(object):
    def __init__(self, file_name=ATOM_TYPES_PATH):
        self.atom_types = []
        self.atom_symbols = []
        with open(file_name, 'r') as f:
            for line in f:
                atom_type_i = line.strip('\n').split(',')
                self.atom_types.append((atom_type_i[0], int(atom_type_i[1]), int(atom_type_i[2])))
                if atom_type_i[0] not in self.atom_symbols:
                    self.atom_symbols.append(atom_type_i[0])

        self.bond_orders = [Chem.BondType.AROMATIC, Chem.BondType.SINGLE,
                            Chem.BondType.DOUBLE, Chem.BondType.TRIPLE]
        self.max_iter = MAX_ITER

    def get_atom_type(self, atom):
        return self.atom_types.index(
            (atom.GetSymbol(), atom.GetFormalCharge(), atom.GetNumExplicitHs()))

    def get_bond_type(self, bond):
        return self.bond_orders.index(bond.GetBondType())

    def index_to_atom(self, idx):
        symbol, charge, hs = self.atom_types[idx]
        a = Chem.Atom(symbol)
        a.SetFormalCharge(charge)
        a.SetNumExplicitHs(hs)
        return a

    def index_to_bond(self, mol, begin_id, end_id, idx):
        mol.AddBond(begin_id, end_id, self.bond_orders[idx])

    @property
    def num_atom_types(self):
        return len(self.atom_types)

    @property
    def num_bond_types(self):
        return len(self.bond_orders)


_mol_spec = None


def get_mol_spec():
    global _mol_spec
    if _mol_spec is None:
        _mol_spec = MoleculeSpec(ATOM_TYPES_PATH)
    return _mol_spec


def get_graph_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES string: {smiles}")

    atom_types, atom_ranks, bonds, bond_types = [], [], [], []
    for a, r in zip(mol.GetAtoms(), Chem.CanonicalRankAtoms(mol)):
        atom_types.append(get_mol_spec().get_atom_type(a))
        atom_ranks.append(r)

    for b in mol.GetBonds():
        idx_1, idx_2 = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bt = get_mol_spec().get_bond_type(b)
        bonds.append([idx_1, idx_2])
        bond_types.append(bt)

    graph = nx.Graph()
    graph.add_nodes_from(range(len(atom_types)))
    graph.add_edges_from(bonds)
    return graph, atom_types, atom_ranks, bonds, bond_types


def traverse_graph(graph, atom_ranks, current_node=None, step_ids=None, p=P_ALPHA, log_p=0.0):
    """Q_alpha proposal: canonical rank with probability p, random neighbour otherwise."""
    if current_node is None:
        next_nodes = range(len(atom_ranks))
        step_ids = [-1] * len(next_nodes)
        next_node_ranks = atom_ranks
    else:
        next_nodes = list(graph.neighbors(current_node))
        next_nodes = [n for n in next_nodes if step_ids[n] < 0]
        next_node_ranks = [atom_ranks[n] for n in next_nodes]

    next_nodes = [n for n, r in sorted(zip(next_nodes, next_node_ranks), key=lambda x: x[1])]

    while len(next_nodes) > 0:
        if len(next_nodes) == 1:
            next_node = next_nodes[0]
        elif random.random() >= (1.0 - p):
            next_node = next_nodes[0]
            log_p += np.log(p)
        else:
            next_node = next_nodes[random.randint(1, len(next_nodes) - 1)]
            log_p += np.log((1.0 - p) / (len(next_nodes) - 1))

        step_ids[next_node] = max(step_ids) + 1
        _, log_p = traverse_graph(graph, atom_ranks, next_node, step_ids, p, log_p)
        next_nodes = [n for n in graph.neighbors(next_node) if step_ids[n] < 0]
        next_node_ranks = [atom_ranks[n] for n in next_nodes]
        next_nodes = [n for n, r in sorted(zip(next_nodes, next_node_ranks), key=lambda x: x[1])]

    return step_ids, log_p


def single_reorder(X_0, A_0, step_ids):
    X_0, A_0 = np.copy(X_0), np.copy(A_0)
    step_ids = np.array(step_ids, dtype=np.int32)
    sorted_ids = np.argsort(step_ids)
    X_0 = X_0[sorted_ids]
    A_0[:, 0], A_0[:, 1] = step_ids[A_0[:, 0]], step_ids[A_0[:, 1]]

    max_b, min_b = np.amax(A_0[:, :2], axis=1), np.amin(A_0[:, :2], axis=1)
    A_0 = A_0[np.lexsort([-min_b, max_b]), :]

    max_b, min_b = np.amax(A_0[:, :2], axis=1), np.amin(A_0[:, :2], axis=1)
    is_append = np.concatenate([np.array([True]), max_b[1:] > max_b[:-1]])
    A_0 = np.concatenate([
        np.where(is_append[:, np.newaxis], np.stack([min_b, max_b], axis=1),
                 np.stack([max_b, min_b], axis=1)),
        A_0[:, -1:]
    ], axis=1)
    return X_0, A_0


def single_expand(X_0, A_0):
    X_0, A_0 = np.copy(X_0), np.copy(A_0)
    is_append_iter = np.less(A_0[:, 0], A_0[:, 1]).astype(np.int32)

    NX = np.cumsum(np.pad(is_append_iter, [[1, 0]], mode='constant', constant_values=1))
    shift = np.cumsum(np.pad(NX, [[1, 0]], mode='constant')[:-1])
    X_index = np.arange(NX.sum(), dtype=np.int32) - np.repeat(shift, NX)
    X = X_0[X_index]

    _, A_index = np.tril_indices(A_0.shape[0])
    A = A_0[A_index, :]
    NA = np.arange(A_0.shape[0] + 1)

    action_type = 1 - is_append_iter
    atom_type = np.where(action_type == 0, X_0[A_0[:, 1]], 0)
    bond_type = A_0[:, 2]
    append_pos = np.where(action_type == 0, A_0[:, 0], 0)
    connect_pos = np.where(action_type == 1, A_0[:, 1], 0)
    actions = np.stack([action_type, atom_type, bond_type, append_pos, connect_pos], axis=1)
    actions = np.append(actions, [[2, 0, 0, 0, 0]], axis=0)
    action_0 = np.array([X_0[0]], dtype=np.int32)

    last_atom_index = shift + NX - 1
    last_atom_mask = np.zeros_like(X)
    last_atom_mask[last_atom_index] = np.where(
        np.pad(is_append_iter, [[1, 0]], mode='constant', constant_values=1) == 1,
        np.ones_like(last_atom_index),
        np.ones_like(last_atom_index) * 2
    )
    return action_0, X, NX, A, NA, actions, last_atom_mask


def get_d(A, X):
    def _to_sparse(_A, _X):
        return sparse.coo_matrix(
            (np.ones([_A.shape[0] * 2], dtype=np.int32),
             (np.concatenate([_A[:, 0], _A[:, 1]], axis=0),
              np.concatenate([_A[:, 1], _A[:, 0]], axis=0))),
            shape=[_X.shape[0]] * 2)

    A_sparse = _to_sparse(A, X)
    d2 = A_sparse * A_sparse
    d3 = d2 * A_sparse

    D_2 = np.stack(d2.nonzero(), axis=1)
    D_2 = D_2[D_2[:, 0] < D_2[:, 1], :]

    D_3 = np.stack(d3.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]

    D_3_sparse = _to_sparse(D_3, X)
    D_3_sparse = D_3_sparse - D_3_sparse.multiply(A_sparse)
    D_3 = np.stack(D_3_sparse.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]
    return D_2, D_3


def merge_single_0(X_0, A_0, NX_0, NA_0):
    cumsum = np.cumsum(np.pad(NX_0, [[1, 0]], mode='constant')[:-1])
    A_0[:, :2] += np.stack([np.repeat(cumsum, NA_0)] * 2, axis=1)
    D_0_2, D_0_3 = get_d(A_0, X_0)

    A_split = []
    for i in range(get_mol_spec().num_bond_types):
        A_split.append(A_0[A_0[:, 2] == i, :2])
    A_split.extend([D_0_2, D_0_3])
    A_0 = A_split
    NX_rep_0 = np.repeat(np.arange(NX_0.shape[0]), NX_0)
    return X_0, A_0, NX_0, NX_rep_0


def merge_single(X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions,
                 last_append_mask, log_p):
    X, A, NX, NX_rep = merge_single_0(X, A, NX, NA)
    cumsum = np.cumsum(np.pad(NX, [[1, 0]], mode='constant')[:-1])
    actions[:, -2] += cumsum * (actions[:, 0] == 0)
    actions[:, -1] += cumsum * (actions[:, 0] == 1)
    mol_ids_rep = np.repeat(mol_ids, NX)
    rep_ids_rep = np.repeat(rep_ids, NX)
    return (X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep,
            action_0, actions, log_p)

## Pair assembly

`ligands_*_corrected` and `proteins_*_corrected` are aligned line-for-line, both
space-separated. `<fold>_<tr|te>_unique_proteins.txt` gives the protein order that matches
`>p1…>pN` in the bepler FASTA and so the HDF5 keys `p1…pN`.

In [ ]:
def read_spaced(path, limit=None):
    out = []
    with open(path) as f:
        for line in f:
            s = ''.join(line.split())
            if s:
                out.append(s)
            if limit is not None and len(out) >= limit:
                break
    return out


def assemble_pairs(split, limit=None):
    """-> (records, C) where records = [(smiles, protein_row, ames), ...]"""
    lig_path, prot_path, uniq_path, emb_path = find_fold_files(split)
    print(f"[{split}] ligands   {os.path.basename(lig_path)}")
    print(f"[{split}] proteins  {os.path.basename(prot_path)}")
    print(f"[{split}] unique    {os.path.basename(uniq_path)}")
    print(f"[{split}] embedding {os.path.basename(emb_path)}")

    ligands = read_spaced(lig_path, limit)
    proteins = read_spaced(prot_path, limit)
    if len(ligands) != len(proteins):
        raise ValueError(f"{split}: {len(ligands)} ligand lines vs {len(proteins)} protein "
                         f"lines — the pair files are not aligned.")

    unique = read_spaced(uniq_path)
    if not unique:
        raise ValueError(f"{uniq_path}: no protein sequences read.")
    seq_to_row = {s: i for i, s in enumerate(unique)}

    global N_C
    C = load_embeddings(emb_path, len(unique))
    N_C = int(C.shape[1])          # only differs from 6165 if ALLOW_N_C_OVERRIDE
    print(f"[{split}] {len(ligands)} pairs | {len(unique)} unique proteins | "
          f"embeddings {C.shape} (one {C.shape[1]}-d vector per protein)")

    # one parse per pair: validity, size cap, atom-type vocabulary
    rows, n_bad, n_big, n_oov, n_unmapped = [], 0, 0, 0, 0
    for smi, seq in zip(ligands, proteins):
        row = seq_to_row.get(seq)
        if row is None:
            n_unmapped += 1
            continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            n_bad += 1
            continue
        if mol.GetNumAtoms() > MAX_ATOMS:
            n_big += 1
            continue
        try:
            for a in mol.GetAtoms():
                get_mol_spec().get_atom_type(a)
        except ValueError:
            n_oov += 1
            continue
        rows.append((smi, row))

    print(f"[{split}] kept {len(rows)} | dropped: unparseable {n_bad}, "
          f">{MAX_ATOMS} atoms {n_big}, out-of-vocab atoms {n_oov}, "
          f"unmatched protein {n_unmapped}")
    if n_unmapped:
        print(f"[WARN] {n_unmapped} pairs had a protein absent from {os.path.basename(uniq_path)}.")
    if not rows:
        raise RuntimeError(f"{split}: no usable pairs.")

    labels = ames_labels([smi for smi, _ in rows])
    records = [(smi, row, lab) for (smi, row), lab in zip(rows, labels)]
    return records, C


def split_by_protein(records, n_proteins, val_fraction, seed):
    """Partition held-out pairs by protein so validation and test share no protein."""
    rng = np.random.default_rng(seed)
    order = rng.permutation(n_proteins)
    n_val = max(1, int(round(val_fraction * n_proteins)))
    val_rows = set(order[:n_val].tolist())

    val = [r for r in records if r[1] in val_rows]
    test = [r for r in records if r[1] not in val_rows]
    return val, test, sorted(val_rows), sorted(set(order[n_val:].tolist()))

## Sampler and collator

In [ ]:
class BalancedSampler(Sampler):
    """Groups similar-length molecules into a batch to cut padding in the GCN/RNN."""
    def __init__(self, cost, batch_size):
        index = np.argsort(cost).tolist()
        chunk_size = int(float(len(cost)) / batch_size)
        self.index = []
        for i in range(batch_size):
            self.index.append(index[i * chunk_size:(i + 1) * chunk_size])

    def _g(self):
        for index_i in self.index:
            random.shuffle(index_i)
        for batch_index in zip(*self.index):
            yield list(batch_index)

    def __iter__(self):
        return self._g()

    def __len__(self):
        return len(self.index[0])


def process_single(record, k=K_PATHS, p=P_ALPHA):
    """Build the graph, sample K traversals, attach the ligand's Ames label."""
    smiles, tox_label = record[0], int(record[2])

    graph, atom_types, atom_ranks, bonds, bond_types = get_graph_from_smiles(smiles)
    X_0 = np.array(atom_types, dtype=np.int32)
    A_0 = np.concatenate([
        np.array(bonds, dtype=np.int32),
        np.array(bond_types, dtype=np.int32)[:, np.newaxis]
    ], axis=1) if len(bonds) > 0 else np.zeros((0, 3), dtype=np.int32)

    X, A, NX, NA = [], [], [], []
    mol_ids, rep_ids, iw_ids = [], [], []
    action_0, actions, last_append_mask = [], [], []
    log_p = []

    tox_class = np.full((k,), tox_label, dtype=np.int32)

    for i in range(k):
        step_ids_i, log_p_i = traverse_graph(graph, atom_ranks, p=p)
        X_i, A_i = single_reorder(X_0, A_0, step_ids_i)
        action_0_i, X_i, NX_i, A_i, NA_i, actions_i, last_atom_mask_i = single_expand(X_i, A_i)

        X.append(X_i)
        A.append(A_i)
        NX.append(NX_i)
        NA.append(NA_i)
        action_0.append(action_0_i)
        actions.append(actions_i)
        last_append_mask.append(last_atom_mask_i)
        mol_ids.append(np.zeros_like(NX_i, dtype=np.int32))
        rep_ids.append(np.ones_like(NX_i, dtype=np.int32) * i)
        iw_ids.append(np.ones_like(NX_i, dtype=np.int32) * i)
        log_p.append(log_p_i)

    return (np.concatenate(X, axis=0), np.concatenate(A, axis=0),
            np.concatenate(NX, axis=0), np.concatenate(NA, axis=0),
            np.concatenate(mol_ids, axis=0), np.concatenate(rep_ids, axis=0),
            np.concatenate(iw_ids, axis=0), np.concatenate(action_0, axis=0),
            np.concatenate(actions, axis=0), np.concatenate(last_append_mask, axis=0),
            np.array(log_p, dtype=np.float32), tox_class)


class CMolRNNCollate:
    """Conditional collator. `c` is gathered for surviving records only, keeping its rows
    aligned with mol_ids."""
    def __init__(self, c_matrix, k=K_PATHS, p=P_ALPHA):
        self.C = c_matrix
        self.k = k
        self.p = p

    def __call__(self, batch):
        valid_batch = []
        for record in batch:
            try:
                _ = get_graph_from_smiles(record[0])
                valid_batch.append(record)
            except ValueError:
                continue
        if not valid_batch:
            return None

        shapes = [[0], [0, 3], [0], [0], [0], [0], [0], [0], [0, 5], [0], [0], [0]]
        dtypes = [np.int32] * 10 + [np.float32, np.int32]

        _build = lambda: [np.zeros(shape=s, dtype=d) for s, d in zip(shapes, dtypes)]
        _append = lambda r0, r1: [np.concatenate([a, b], axis=0) for a, b in zip(r0, r1)]

        (X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions,
         last_append_mask, log_p, tox_class) = _build()

        for n, record in enumerate(valid_batch):
            (X_i, A_i, NX_i, NA_i, mol_ids_i, rep_ids_i, iw_ids_i, action_0_i,
             actions_i, last_append_mask_i, log_p_i, tox_class_i) = process_single(
                record, self.k, self.p)

            if n != 0:
                mol_ids_i += mol_ids[-1] + 1
                iw_ids_i += iw_ids[-1] + 1

            (X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions,
             last_append_mask, log_p, tox_class) = _append(
                [X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions,
                 last_append_mask, log_p, tox_class],
                [X_i, A_i, NX_i, NA_i, mol_ids_i, rep_ids_i, iw_ids_i, action_0_i,
                 actions_i, last_append_mask_i, log_p_i, tox_class_i.reshape(-1)])

        (X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep,
         action_0, actions, log_p) = merge_single(
            X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p)

        result_out = [X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep,
                      action_0, actions, log_p, tox_class]

        mol_list = [Chem.MolFromSmiles(b[0]) for b in valid_batch]
        graph_to_rnn = np.zeros((len(valid_batch), self.k, get_mol_spec().max_iter),
                                dtype=np.int32)
        rnn_to_graph = []
        cum_sum = 0
        for i, mol_i in enumerate(mol_list):
            num_iter = mol_i.GetNumBonds() + 1
            for k_idx in range(self.k):
                graph_to_rnn[i, k_idx, :num_iter] = np.arange(num_iter) + cum_sum
                r0 = np.ones([num_iter], dtype=np.int32) * i
                r1 = np.ones_like(r0) * k_idx
                r2 = np.arange(num_iter)
                rnn_to_graph.append(np.stack([r0, r1, r2], axis=0))
                cum_sum += num_iter
        rnn_to_graph = np.concatenate(rnn_to_graph, axis=1)
        NX_cum = np.cumsum(result_out[6])

        c = self.C[[rec[1] for rec in valid_batch], :]
        return result_out + [graph_to_rnn, rnn_to_graph, NX_cum, c]


def batch_to_tensor(record, device=None):
    """Move one collated batch to the device. `ids` maps each atom row to its molecule, which
    is how `c` is broadcast over atoms."""
    device = CTX if device is None else device
    *base, graph_to_rnn, rnn_to_graph, NX_cum, c = record
    (X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask,
     NX, NX_rep, action_0, actions, log_p, tox_class) = base

    n_nodes = int(X.shape[0])
    X = torch.as_tensor(X, dtype=torch.long, device=device)

    A_sparse = []
    for A_i in A:
        if A_i.shape[0] == 0:
            A_sparse.append(None)
        else:
            A_i = np.concatenate([A_i, A_i[:, [1, 0]]], axis=0)
            idx = torch.as_tensor(A_i[:, :2].T.astype(np.int64), device=device)
            val = torch.ones(A_i.shape[0], dtype=torch.float32, device=device)
            A_sparse.append(torch.sparse_coo_tensor(
                idx, val, (n_nodes, n_nodes), dtype=torch.float32, device=device).coalesce())

    batch_size = int(mol_ids_rep.max() + 1)
    iw_size = int(rep_ids_rep.max() + 1)

    ids = torch.as_tensor(mol_ids_rep, dtype=torch.long, device=device)
    iw_ids, last_append_mask, NX, NX_rep, action_0, actions = [
        torch.as_tensor(x, dtype=torch.long, device=device)
        for x in [iw_ids, last_append_mask, NX, NX_rep, action_0, actions]]
    log_p = torch.as_tensor(log_p, dtype=torch.float32, device=device)
    tox_class = torch.as_tensor(tox_class, dtype=torch.float32, device=device)
    graph_to_rnn = torch.as_tensor(graph_to_rnn, dtype=torch.long, device=device)
    rnn_to_graph = torch.as_tensor(rnn_to_graph, dtype=torch.long, device=device)
    NX_cum = torch.as_tensor(NX_cum, dtype=torch.long, device=device)
    c = torch.as_tensor(np.ascontiguousarray(c), dtype=torch.float32, device=device)

    return [X, A_sparse, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p,
            batch_size, iw_size, tox_class, graph_to_rnn, rnn_to_graph, NX_cum, c, ids]

## Build datasets

Train pairs come from `train_dataset`. The held-out `test_dataset` proteins are split in two:
one half drives early stopping, the other is saved untouched to `test_set.npz` for evaluation.

In [ ]:
records_train, C_train = assemble_pairs("train", LIMIT_PAIRS)
records_held, C_held = assemble_pairs("test", LIMIT_PAIRS)

records_val, records_test, val_rows, test_rows = split_by_protein(
    records_held, C_held.shape[0], VAL_PROTEIN_FRACTION, SPLIT_SEED)

assert C_train.shape[1] == C_held.shape[1], (
    f"train embeddings are {C_train.shape[1]}-d but held-out are {C_held.shape[1]}-d")

print(f"\ntrain {len(records_train)} pairs / {C_train.shape[0]} proteins")
print(f"val   {len(records_val)} pairs / {len(val_rows)} proteins")
print(f"test  {len(records_test)} pairs / {len(test_rows)} proteins  (held out)")

with open(SPLIT_MANIFEST, 'w') as f:
    json.dump({'fold': FOLD, 'fold_suffix': FOLD_SUFFIX[FOLD], 'N_C': N_C,
               'split_seed': SPLIT_SEED, 'val_protein_fraction': VAL_PROTEIN_FRACTION,
               'val_protein_rows': val_rows, 'test_protein_rows': test_rows,
               'n_train_pairs': len(records_train), 'n_val_pairs': len(records_val),
               'n_test_pairs': len(records_test)}, f, indent=1)

np.savez_compressed(
    TEST_BUNDLE,
    smiles=np.array([r[0] for r in records_test], dtype=object),
    protein_row=np.array([r[1] for r in records_test], dtype=np.int32),
    ames=np.array([r[2] for r in records_test], dtype=np.int8),
    embeddings=C_held,
    test_protein_rows=np.array(test_rows, dtype=np.int32))
print(f"[OK] test bundle -> {TEST_BUNDLE}")

# PORT: upstream `Lambda.__init__` shuffled with seed 17.
random.Random(17).shuffle(records_train)

loader_train = DataLoader(
    records_train,
    batch_sampler=BalancedSampler([len(r[0]) for r in records_train], BATCH_SIZE),
    num_workers=NUM_WORKERS,
    collate_fn=CMolRNNCollate(C_train, K_PATHS, P_ALPHA),
    persistent_workers=(NUM_WORKERS > 0))

loader_val = DataLoader(
    records_val,
    batch_sampler=BalancedSampler([len(r[0]) for r in records_val], BATCH_SIZE_VAL),
    num_workers=NUM_WORKERS,
    collate_fn=CMolRNNCollate(C_held, K_PATHS, P_ALPHA),
    persistent_workers=(NUM_WORKERS > 0))

batch_cpu = next(iter(loader_train))
_b = batch_to_tensor(batch_cpu)
print(f"\nbatch: N={_b[9]} K={_b[10]} | X {tuple(_b[0].shape)} | "
      f"A blocks {len(_b[1])} | c {tuple(_b[15].shape)}")
assert _b[15].shape == (_b[9], N_C)
assert int(_b[16].max()) == _b[9] - 1

## Graph math

In [ ]:
def segment_sum(X, idx, num_seg):
    out = torch.zeros((num_seg,) + tuple(X.shape[1:]), dtype=X.dtype, device=X.device)
    return out.index_add(0, idx, X)


def _concat_neighbourhoods(X, A_list):
    parts = [X]
    for A in A_list:
        parts.append(torch.sparse.mm(A, X) if A is not None else torch.zeros_like(X))
    return torch.cat(parts, dim=1)


class EfficientGraphConvFn(torch.autograd.Function):
    """Recomputes the concatenated neighbourhood in backward instead of storing it.
    Every A is symmetric, so A^T == A and backward reuses A directly."""

    @staticmethod
    def forward(ctx, X, W, A_list):
        ctx.A_list = A_list
        ctx.save_for_backward(X, W)
        return _concat_neighbourhoods(X, A_list) @ W

    @staticmethod
    @once_differentiable
    def backward(ctx, grad_output):
        X, W = ctx.saved_tensors
        A_list = ctx.A_list
        X_out = _concat_neighbourhoods(X, A_list)
        grad_output = grad_output.contiguous()

        grad_W = X_out.t() @ grad_output
        grad_X_out = grad_output @ W.t()

        chunks = torch.split(grad_X_out, X.shape[1], dim=1)
        grad_X = chunks[0]
        for A, g in zip(A_list, chunks[1:]):
            if A is not None:
                grad_X = grad_X + torch.sparse.mm(A, g.contiguous())
        return grad_X, grad_W, None


def get_activation(name):
    return {'relu': torch.relu, 'tanh': torch.tanh}[name]

## Layers

In [ ]:
class Linear_BN(nn.Sequential):
    def __init__(self, F_in, F_out):
        super(Linear_BN, self).__init__(
            nn.Linear(F_in, F_out, bias=False),
            nn.BatchNorm1d(F_out, eps=1e-5, momentum=0.1))


class GraphConv(nn.Module):
    def __init__(self, Fin, Fout, D):
        super(GraphConv, self).__init__()
        self.Fin, self.Fout, self.D = Fin, Fout, D
        self.W = nn.Parameter(torch.empty(self.Fin * (self.D + 1), self.Fout))

    def forward(self, X, A_list):
        return EfficientGraphConvFn.apply(X, self.W, A_list)


class Policy(nn.Module):
    def __init__(self, F_in, F_h, N_A, N_B, k=1):
        super(Policy, self).__init__()
        self.F_in, self.F_h, self.N_A, self.N_B, self.k = F_in, F_h, N_A, N_B, k
        self.linear_h = Linear_BN(F_in * 2, self.F_h * k)
        self.linear_h_t = Linear_BN(F_in, self.F_h * k)
        self.linear_x = nn.Linear(self.F_h, self.N_B + self.N_B * self.N_A)
        self.linear_x_t = nn.Linear(self.F_h, 1)
        self.linear_pi = nn.Linear(self.F_in, self.k) if self.k > 1 else None

    def forward(self, X, NX, NX_rep, X_end=None):
        if X_end is None:
            X_end = segment_sum(X, NX_rep, NX.shape[0]) / NX.unsqueeze(1).float()
        X = torch.cat([X, X_end[NX_rep, :]], dim=1)

        X_h = torch.relu(self.linear_h(X)).reshape(-1, self.F_h)
        X_h_end = torch.relu(self.linear_h_t(X_end)).reshape(-1, self.F_h)

        # clip before exp() to stay inside float32, as in the original
        logits_x = torch.clamp(
            self.linear_x(X_h).reshape(-1, self.k, self.N_B + self.N_B * self.N_A), -80.0, 80.0)
        logits_x_t = torch.clamp(
            self.linear_x_t(X_h_end).reshape(-1, self.k, 1), -80.0, 80.0)

        X_x = torch.exp(logits_x)
        X_x_end = torch.exp(logits_x_t)

        # one joint softmax over every action of every atom plus `end`, per molecule
        X_sum = segment_sum(X_x, NX_rep, NX.shape[0]).sum(-1, keepdim=True) + X_x_end
        X_softmax = X_x / (X_sum[NX_rep, :, :] + 1e-10)
        X_softmax_end = X_x_end / (X_sum + 1e-10)

        if self.k > 1:
            pi = torch.softmax(self.linear_pi(X_end), dim=1).unsqueeze(-1)
            X_softmax = (X_softmax * pi[NX_rep, :, :]).sum(dim=1)
            X_softmax_end = (X_softmax_end * pi).sum(dim=1)
        else:
            X_softmax = X_softmax.squeeze(1)
            X_softmax_end = X_softmax_end.squeeze(1)

        connect, append = X_softmax[:, :self.N_B], X_softmax[:, self.N_B:]
        append = append.reshape(-1, self.N_A, self.N_B)
        return append, connect, X_softmax_end.squeeze(-1)

## Generator

Parameter names outside `linear_c.*` and `dense_policy_0.*` match the pretraining model, so
the phase-1 checkpoint transfers by name.

In [ ]:
class MoleculeGenerator(nn.Module):
    __metaclass__ = ABCMeta

    def __init__(self, N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, *args, **kwargs):
        super(MoleculeGenerator, self).__init__()
        self.N_A, self.N_B, self.D = N_A, N_B, D
        self.F_e, self.F_skip, self.Fh_policy = F_e, F_skip, Fh_policy
        self.F_c = list(F_c)
        self.activation = get_activation(activation)

        self.embedding_atom = nn.Embedding(self.N_A, self.F_e)
        self.embedding_mask = nn.Embedding(3, self.F_e)
        self._build_graph_conv(*args, **kwargs)

        self.dense = nn.Sequential()
        for i, (f_in, f_out) in enumerate(zip([self.F_skip] + self.F_c[:-1], self.F_c)):
            self.dense.add_module(str(i), Linear_BN(f_in, f_out))

        self.policy_0 = nn.Parameter(torch.zeros(self.N_A))
        self.policy_h = Policy(self.F_c[-1], self.Fh_policy, self.N_A, self.N_B)
        self.mode = 'loss'
        self.last_l = None          # per-branch logging
        self.last_tox = None

    @abstractmethod
    def _build_graph_conv(self, *args, **kwargs): pass

    def _likelihood(self, init, append, connect, end, action_0, actions, iw_ids,
                    log_p_sigma, batch_size, iw_size, tox_class_batch):
        action_type, node_type, edge_type, append_pos, connect_pos = (
            actions[:, 0], actions[:, 1], actions[:, 2], actions[:, 3], actions[:, 4])

        def _log_mask(_x, _mask):
            return _mask * torch.log(_x + 1e-10) + (1 - _mask) * torch.zeros_like(_x)

        init = init.reshape(batch_size * iw_size, self.N_A)
        rows = torch.arange(action_0.shape[0], device=action_0.device)
        loss_init = torch.log(init[rows, action_0] + 1e-10)

        loss_end = _log_mask(end, (action_type == 2).to(end.dtype))
        loss_append = _log_mask(append[append_pos, node_type, edge_type],
                                (action_type == 0).to(end.dtype))
        loss_connect = _log_mask(connect[connect_pos, edge_type],
                                 (action_type == 1).to(end.dtype))

        log_p_x = loss_end + loss_append + loss_connect
        log_p_x = segment_sum(log_p_x.unsqueeze(-1), iw_ids, batch_size * iw_size).squeeze(-1)
        log_p_x = log_p_x + loss_init

        log_p_x = log_p_x.reshape(batch_size, iw_size)
        log_p_sigma = log_p_sigma.reshape(batch_size, iw_size)

        l = log_p_x - log_p_sigma
        l = torch.logsumexp(l, dim=1) - math.log(float(iw_size))

        # Eq. 8: all K paths share the molecule's label, so column 0 is the label
        tox_labels = tox_class_batch.reshape(batch_size, iw_size)[:, 0]
        if TOX_FLOOR is not None:
            l = torch.where(tox_labels > 0, torch.clamp(l, min=TOX_FLOOR), l)
        l_weighted = l * (1.0 - (1.0 + LAMBDA_TOX) * tox_labels)

        self.last_l = l.detach()
        self.last_tox = tox_labels.detach()
        return l_weighted


class MoleculeGenerator_RNN(MoleculeGenerator):
    __metaclass__ = ABCMeta

    def __init__(self, N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn,
                 *args, **kwargs):
        super(MoleculeGenerator_RNN, self).__init__(
            N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, *args, **kwargs)
        self.N_rnn = N_rnn
        self.rnn = nn.GRU(input_size=self.F_c[-1] * 2, hidden_size=self.F_c[-1],
                          num_layers=self.N_rnn, batch_first=True)

    def _rnn_train(self, X, NX, NX_rep, graph_to_rnn, rnn_to_graph, NX_cum):
        X_avg = segment_sum(X, NX_rep, NX.shape[0]) / NX.unsqueeze(1).float()
        X = torch.cat([X_avg, X[NX_cum - 1]], dim=1)
        X = X[graph_to_rnn]
        batch_size, iw_size, length, num_features = X.shape
        out, _ = self.rnn(X.reshape(batch_size * iw_size, length, num_features))
        out = out.reshape(batch_size, iw_size, length, -1)
        return out[rnn_to_graph[0], rnn_to_graph[1], rnn_to_graph[2]]

    def _rnn_test(self, X, NX, NX_rep, NX_cum, h):
        X_avg = segment_sum(X, NX_rep, NX.shape[0]) / NX.unsqueeze(1).float()
        X = torch.cat([X_avg, X[NX_cum - 1]], dim=1)
        X, h = self.rnn(X.unsqueeze(1), h)
        return X.squeeze(1), h


class _TwoLayerDense(nn.Module):
    """FNN_0: protein embedding -> first-atom distribution."""
    def __init__(self, input_size, hidden_size, output_size):
        super(_TwoLayerDense, self).__init__()
        self.input = nn.Linear(input_size, hidden_size, bias=False)
        self.bn_input = nn.BatchNorm1d(hidden_size, eps=1e-5, momentum=0.1)
        self.output = nn.Linear(hidden_size, output_size, bias=True)

    def forward(self, c):
        return torch.softmax(self.output(torch.relu(self.bn_input(self.input(c)))), dim=-1)


class CMoleculeGenerator_RNN(MoleculeGenerator_RNN):
    __metaclass__ = ABCMeta

    def __init__(self, N_A, N_B, N_C, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn,
                 *args, **kwargs):
        self.N_C = N_C
        super(CMoleculeGenerator_RNN, self).__init__(
            N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn, *args, **kwargs)
        self.dense_policy_0 = _TwoLayerDense(self.N_C, self.N_A * 3, self.N_A)

    @abstractmethod
    def _graph_conv_forward(self, X, A, c, ids): pass

    def _policy_0(self, c):
        # the 0.0 * policy_0 term keeps the unused pretraining parameter in the graph
        return self.dense_policy_0(c) + 0.0 * self.policy_0

    def _policy(self, X, A, NX, NX_rep, last_append_mask,
                graph_to_rnn, rnn_to_graph, NX_cum, c, ids):
        X = self.embedding_atom(X) + self.embedding_mask(last_append_mask)
        X = self._graph_conv_forward(X, A, c, ids)
        X_dense = self.dense(X)           # PORT: computed once, not twice as in MXNet
        X_mol = self._rnn_train(X_dense, NX, NX_rep, graph_to_rnn, rnn_to_graph, NX_cum)
        return self.policy_h(X_dense, NX, NX_rep, X_mol)

    def _decode_step(self, X, A, NX, NX_rep, last_append_mask, NX_cum, h, c, ids):
        """One sampling step — unused in training, needed by the sampler."""
        X = self.embedding_atom(X) + self.embedding_mask(last_append_mask)
        X = self._graph_conv_forward(X, A, c, ids)
        X_dense = self.dense(X)
        X_mol, h = self._rnn_test(X_dense, NX, NX_rep, NX_cum, h)
        append, connect, end = self.policy_h(X_dense, NX, NX_rep, X_mol)
        return append, connect, end, h

    def forward(self, *inputs):
        if self.mode in ('loss', 'likelihood'):
            (X, A, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p,
             batch_size, iw_size, tox_class, graph_to_rnn, rnn_to_graph, NX_cum, c, ids) = inputs
            init = self._policy_0(c).unsqueeze(1).expand(-1, iw_size, -1)
            append, connect, end = self._policy(
                X, A, NX, NX_rep, last_append_mask, graph_to_rnn, rnn_to_graph, NX_cum, c, ids)
            l_weighted = self._likelihood(init, append, connect, end, action_0, actions,
                                          iw_ids, log_p, batch_size, iw_size, tox_class)
            return l_weighted if self.mode == 'likelihood' else -l_weighted.mean()
        elif self.mode == 'decode_0':
            return self._policy_0(*inputs)
        elif self.mode == 'decode_step':
            return self._decode_step(*inputs)
        raise ValueError(f"Unknown mode: {self.mode}")


class CVanillaMolGen_RNN(CMoleculeGenerator_RNN):
    def __init__(self, N_A, N_B, N_C, D, F_e, F_h, F_skip, F_c, Fh_policy, activation, N_rnn):
        super(CVanillaMolGen_RNN, self).__init__(
            N_A, N_B, N_C, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn, F_h)

    def _build_graph_conv(self, F_h):
        self.F_h = list(F_h)
        self.conv = nn.ModuleList()
        self.bn = nn.ModuleList()            # bn[i-1] belongs to conv[i]; conv[0] has none
        for i, (f_in, f_out) in enumerate(zip([self.F_e] + self.F_h[:-1], self.F_h)):
            self.conv.append(GraphConv(f_in, f_out, self.N_B + self.D))
            if i != 0:
                self.bn.append(nn.BatchNorm1d(f_in, eps=1e-5, momentum=0.1))

        self.bn_skip = nn.BatchNorm1d(sum(self.F_h), eps=1e-5, momentum=0.1)
        self.linear_skip = Linear_BN(sum(self.F_h), self.F_skip)
        # FNN_1..FNN_l: the per-layer protein projectors
        self.linear_c = nn.ModuleList(
            [nn.Linear(self.N_C, f_out, bias=False) for f_out in self.F_h])

    def _graph_conv_forward(self, X, A, c, ids):
        X_out = [X]
        for i, conv in enumerate(self.conv):
            X = X_out[-1]
            c_i = self.linear_c[i](c)[ids, :]
            if i == 0:
                X_out.append(conv(X, A) + c_i)
            else:
                X_out.append(conv(self.activation(self.bn[i - 1](X)), A) + c_i)
        X_out = torch.cat(X_out[1:], dim=1)
        return self.activation(self.linear_skip(self.activation(self.bn_skip(X_out))))

## Init and early stopping

In [ ]:
def init_weights_xavier(module):
    """MXNet's default Xavier (uniform, avg, magnitude 3) == xavier_uniform_ with gain 1."""
    for name, p in module.named_parameters():
        if p.dim() < 2 or 'bn' in name.split('.')[-1]:
            continue
        nn.init.xavier_uniform_(p, gain=1.0)
    for name, p in module.named_parameters():
        if 'bias' in name.split('.')[-1]:
            nn.init.zeros_(p)
    for m in module.modules():
        if isinstance(m, nn.BatchNorm1d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
            m.reset_running_stats()
    if hasattr(module, 'policy_0'):
        nn.init.zeros_(module.policy_0)
    return module


class EarlyStopping(object):
    def __init__(self, patience=5, delta=1e-2):
        self.patience, self.delta = patience, delta
        self.best, self.counter = math.inf, 0
        self.improved = False

    def step(self, loss):
        if loss < self.best - self.delta:
            self.best, self.counter, self.improved = loss, 0, True
        else:
            self.counter += 1
            self.improved = False
        return self.counter >= self.patience

    def state_dict(self):
        return {'best': self.best, 'counter': self.counter}

    def load_state_dict(self, s):
        self.best = s.get('best', self.best)
        self.counter = s.get('counter', self.counter)

## Model, weight transfer, resume

In [ ]:
configs = {'N_C': N_C, 'F_e': F_E, 'F_h': F_H, 'F_skip': F_SKIP, 'F_c': F_C,
           'Fh_policy': FH_POLICY, 'activation': ACTIVATION, 'N_rnn': N_RNN}

is_continuous = False
global_counter = 0
epoch_no = 1

if not FORCE_RESTART and all(os.path.isfile(f) for f in (CKPT_LAST, CKPT_TRAINER, CKPT_CONFIG)):
    is_continuous = True
    with open(CKPT_CONFIG) as f:
        configs = json.load(f)
    print("[INFO] Resuming from an existing fine-tuning checkpoint.")
else:
    with open(CKPT_CONFIG, 'w') as f:
        json.dump(configs, f, indent=1)
    print("[INFO] Fresh fine-tuning run.")

model = CVanillaMolGen_RNN(get_mol_spec().num_atom_types, get_mol_spec().num_bond_types,
                           D=D_RECEPTIVE, **configs)


def load_pretrained(model, path):
    """Transfer phase-1 weights. Only the conditioning path may be missing."""
    state = torch.load(path, map_location='cpu', weights_only=False)
    sd = state['model'] if isinstance(state, dict) and 'model' in state else state
    missing, unexpected = model.load_state_dict(sd, strict=False)

    new = ('linear_c.', 'dense_policy_0.')
    unexplained = [k for k in missing if not k.startswith(new)]
    if unexplained:
        raise RuntimeError(f"Pretrained checkpoint missing non-conditioning tensors: "
                           f"{unexplained[:10]} — the two phases have drifted apart.")
    print(f"[INFO] Transferred {len(sd) - len(unexpected)}/{len(sd)} pretrained tensors; "
          f"{len(missing)} conditioning tensors newly initialised.")
    if unexpected:
        print(f"[WARN] Unused tensors in checkpoint: {unexpected[:10]}")


if is_continuous:
    model.load_state_dict(torch.load(CKPT_LAST, map_location='cpu',
                                     weights_only=False)['model'])
else:
    init_weights_xavier(model)
    if os.path.exists(PRETRAINED_CKPT):
        load_pretrained(model, PRETRAINED_CKPT)
    else:
        print("[WARN] Training from random init — not the two-phase protocol.")
model.to(CTX)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.0)
early_stopping = EarlyStopping(PATIENCE, ES_DELTA)

if is_continuous:
    tstate = torch.load(CKPT_TRAINER, map_location='cpu', weights_only=False)
    optimizer.load_state_dict(tstate['optimizer'])
    for g in optimizer.param_groups:
        g['lr'] = tstate.get('lr', LR)
    global_counter = int(tstate.get('global_counter', 0))
    epoch_no = int(tstate.get('epoch_no', 1))
    early_stopping.load_state_dict(tstate.get('early_stopping', {}))
    print(f"[INFO] iteration {global_counter}, epoch {epoch_no}, "
          f"lr {optimizer.param_groups[0]['lr']:.3e}, best {early_stopping.best:.4f}, "
          f"patience {early_stopping.counter}/{PATIENCE}")

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_cond = sum(p.numel() for n, p in model.named_parameters()
             if n.startswith(('linear_c.', 'dense_policy_0.')))
print(f"[INFO] {n_trainable:,} trainable parameters ({n_cond:,} in the conditioning path)")

## Sanity checks

In [ ]:
def sanity_check(model, batch_cpu):
    prev = model.mode
    model.mode = 'likelihood'
    model.eval()
    try:
        with torch.no_grad():
            inputs = batch_to_tensor(batch_cpu)
            lw = model(*inputs)
            l, tox = model.last_l, model.last_tox
        err = (lw - l * (1.0 - (1.0 + LAMBDA_TOX) * tox)).abs().max().item()
        n_tox = int(tox.sum())
        print(f"Eq. 8  | {tox.numel()} molecules, {n_tox} toxic | max error {err:.2e}")
        assert err < 1e-3, "Eq. 8 weighting does not match the closed form."
        if n_tox:
            flipped = int((torch.sign(lw[tox > 0]) == -torch.sign(l[tox > 0])).sum())
            print(f"       | sign flipped on {flipped}/{n_tox} toxic molecules")

        model.mode = 'loss'
        with torch.no_grad():
            a = model(*inputs).item()
            inputs[15] = torch.randn_like(inputs[15]) * 0.1
            b = model(*inputs).item()
        print(f"cond   | real c {a:.4f} vs random c {b:.4f} | delta {abs(a - b):.4f}")
        assert abs(a - b) > 1e-6, "The condition has no effect — conditioning path is dead."
        print("[OK]")
    finally:
        model.mode = prev
        model.train()


sanity_check(model, batch_cpu)

## Training loop

In [ ]:
TRAIN_HEADER = 'step\ttime(h)\tloss\tlr\tl_nontoxic\tl_toxic\tepoch\n'
VAL_HEADER = ('step\ttime(h)\tepoch\tval_weighted\tval_nll\tval_l_nontoxic\tval_l_toxic'
              '\tes_counter\tbest\n')


def _is_data_row(line):
    line = line.strip()
    return bool(line) and not line.startswith(('#', 'step\t'))


if not is_continuous:
    t0 = time.time()
    with open(LOG_FILE, 'w') as f:
        f.write(TRAIN_HEADER)
    with open(VAL_LOG_FILE, 'w') as f:
        f.write(VAL_HEADER)
else:
    for path, header in ((LOG_FILE, TRAIN_HEADER), (VAL_LOG_FILE, VAL_HEADER)):
        if not os.path.isfile(path):
            with open(path, 'w') as f:
                f.write(header)
    last_hours = 0.0
    try:
        with open(LOG_FILE) as f:
            rows = [r.strip() for r in f if _is_data_row(r)]
        if rows:
            last_hours = float(rows[-1].split('\t')[1])
    except (OSError, ValueError, IndexError):
        pass
    t0 = time.time() - last_hours * 3600.0
    print(f"[INFO] Clock resumed at {last_hours:.2f} h.")


def evaluate(model, loader):
    """Molecule-weighted means: (Eq. 8 loss, plain NLL, mean l non-toxic, mean l toxic)."""
    prev = model.mode
    model.mode = 'likelihood'
    model.eval()
    sum_w = sum_l = 0.0
    n_mol = 0
    sum_non, n_non, sum_tox, n_tox = 0.0, 0, 0.0, 0
    try:
        with torch.no_grad():
            for batch in loader:
                if batch is None:
                    continue
                lw = model(*batch_to_tensor(batch))
                l, tox = model.last_l, model.last_tox
                sum_w += float(lw.sum())
                sum_l += float(l.sum())
                n_mol += int(l.numel())
                if (tox == 0).any():
                    sum_non += float(l[tox == 0].sum()); n_non += int((tox == 0).sum())
                if (tox > 0).any():
                    sum_tox += float(l[tox > 0].sum()); n_tox += int((tox > 0).sum())
    finally:
        model.mode = prev
        model.train()
    if n_mol == 0:
        return (float('nan'),) * 4
    return (-sum_w / n_mol, -sum_l / n_mol,
            sum_non / n_non if n_non else float('nan'),
            sum_tox / n_tox if n_tox else float('nan'))


def save_state(best=False):
    torch.save({'model': model.state_dict()}, CKPT_LAST)
    torch.save({'optimizer': optimizer.state_dict(),
                'lr': optimizer.param_groups[0]['lr'],
                'global_counter': global_counter, 'epoch_no': epoch_no,
                'early_stopping': early_stopping.state_dict()}, CKPT_TRAINER)
    if best:
        torch.save({'model': model.state_dict(), 'configs': configs, 'fold': FOLD,
                    'epoch': epoch_no - 1, 'global_counter': global_counter,
                    'val_loss': early_stopping.best, 'N_C': N_C,
                    'lambda_tox': LAMBDA_TOX}, CKPT_BEST)


model.mode = 'loss'
model.train()
it_train = iter(loader_train)
stop = False
prev_val_nll = math.inf

while True:
    global_counter += 1
    end_of_epoch = False

    try:
        inputs_cpu = next(it_train)
    except StopIteration:
        end_of_epoch = True
        print(f"Epoch {epoch_no} complete.")
        epoch_no += 1
        it_train = iter(loader_train)
        inputs_cpu = next(it_train)

    if inputs_cpu is None:
        global_counter -= 1
        continue

    optimizer.zero_grad(set_to_none=True)        # PORT: Gluon overwrites grads, torch accumulates
    loss = model(*batch_to_tensor(inputs_cpu))
    loss.backward()
    torch.nn.utils.clip_grad_value_(model.parameters(), CLIP_GRAD)   # MXNet clip_gradient
    optimizer.step()

    if global_counter % DECAY_STEP == 0:
        for g in optimizer.param_groups:
            g['lr'] *= (1.0 - DECAY)

    if global_counter % SUMMARY_STEP == 0:
        lr_now = optimizer.param_groups[0]['lr']
        hours = (time.time() - t0) / 3600.0
        _l, _tox = model.last_l, model.last_tox
        l_non = _l[_tox == 0].mean().item() if (_tox == 0).any() else float('nan')
        l_tox = _l[_tox > 0].mean().item() if (_tox > 0).any() else float('nan')

        save_state()
        with open(LOG_FILE, 'a') as f:
            f.write(f"{global_counter}\t{hours:.4f}\t{loss.item():.4f}\t{lr_now}"
                    f"\t{l_non:.4f}\t{l_tox:.4f}\t{epoch_no}\n")
        print(f"iter {global_counter:06d} | ep {epoch_no} | loss {loss.item():.4f} | "
              f"l_nontox {l_non:.3f} | l_tox {l_tox:.3f} | lr {lr_now:.3e} | {hours:.2f}h")
        del loss, inputs_cpu
        gc.collect()

    if end_of_epoch:
        val_w, val_nll, val_non, val_tox = evaluate(model, loader_val)
        stop = early_stopping.step(val_w if EARLY_STOP_ON == 'weighted' else val_nll)
        hours = (time.time() - t0) / 3600.0

        with open(VAL_LOG_FILE, 'a') as f:
            f.write(f"{global_counter}\t{hours:.4f}\t{epoch_no - 1}\t{val_w:.4f}"
                    f"\t{val_nll:.4f}\t{val_non:.4f}\t{val_tox:.4f}"
                    f"\t{early_stopping.counter}\t{early_stopping.best:.4f}\n")
        print(f"  val | weighted {val_w:.4f} | nll {val_nll:.4f} | l_nontox {val_non:.3f} | "
              f"l_tox {val_tox:.3f} | best {early_stopping.best:.4f} | "
              f"patience {early_stopping.counter}/{PATIENCE}"
              f"{'  <- new best' if early_stopping.improved else ''}")

        if early_stopping.improved and val_nll > prev_val_nll + 1.0:
            print("  [WARN] weighted loss improved but NLL worsened — the toxic branch may be "
                  "diverging. Consider EARLY_STOP_ON='nll' or TOX_FLOOR=-150.0.")
        prev_val_nll = val_nll

        save_state(best=early_stopping.improved)
        if stop:
            print(f"Early stopping at epoch {epoch_no - 1}.")
            break

    if epoch_no > MAX_EPOCHS:
        print(f"Reached MAX_EPOCHS = {MAX_EPOCHS}.")
        break

save_state()
if not os.path.exists(CKPT_BEST):
    save_state(best=True)       # no epoch improved; keep the final weights as "best"
with open(LOG_FILE, 'a') as f:
    f.write('# finished\n')

print(f"\nDone. Best validation {EARLY_STOP_ON} loss: {early_stopping.best:.4f}")
for name, path in (("best weights", CKPT_BEST), ("last weights", CKPT_LAST),
                   ("configs", CKPT_CONFIG), ("test bundle", TEST_BUNDLE),
                   ("split manifest", SPLIT_MANIFEST), ("train log", LOG_FILE),
                   ("val log", VAL_LOG_FILE)):
    print(f"  {name:15s} {path}")